# Lucas Phase 1 Audit: Continue or Pivot

This notebook implements a strict audit gate for the current cross-country annual project.

Decision outputs:
- `GO_CONTINUE`
- `GO_PIVOT_SHORT_RUN`

All decisions use explicit thresholds only (no manual override).


## Audit Criteria (Hard Rules)

1. Preferred IV first-stage strength: `F >= 10`.
2. Inflation coefficient sign positive in at least 4/5 tracked specs.
3. Inflation coefficient drift across stability checks `< 40%` from baseline FE coefficient.
4. GDP-growth effect of `m2_growth` remains statistically weak across core specs (`p >= 0.05`).
5. Placebo tests do not show significance (`p >= 0.05`).

If any criterion fails: `GO_PIVOT_SHORT_RUN`.


In [ ]:
from pathlib import Path
import subprocess
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path('/Users/stevenchung/Desktop/P12B_File/New project_')
BASE_PATH = ROOT / 'macro_growth_merged.csv'
CTRL_PATH = ROOT / 'data/phase1_controls.csv'
IV_PATH = ROOT / 'data/phase1_instruments.csv'

OUT = ROOT / 'outputs/phase1_audit'
OUT_T = OUT / 'tables'
OUT_F = OUT / 'figures'
OUT_T.mkdir(parents=True, exist_ok=True)
OUT_F.mkdir(parents=True, exist_ok=True)

LOG = []

def log(x):
    print(x)
    LOG.append(str(x))

assert BASE_PATH.exists(), f'Missing {BASE_PATH}'
assert CTRL_PATH.exists(), f'Missing {CTRL_PATH}'
assert IV_PATH.exists(), f'Missing {IV_PATH}'


In [ ]:
base = pd.read_csv(BASE_PATH)
ctrl = pd.read_csv(CTRL_PATH)
iv = pd.read_csv(IV_PATH)

df = base.merge(ctrl, on=['Country Name', 'year'], how='left')
df = df.merge(iv, on=['Country Name', 'year'], how='left')

rows = len(df)
countries = df['Country Name'].nunique()
year_min, year_max = int(df['year'].min()), int(df['year'].max())
dup_country_year = int(df.duplicated(['Country Name', 'year']).sum())

audit_data = pd.DataFrame([
    {'check': 'rows', 'value': rows},
    {'check': 'countries', 'value': countries},
    {'check': 'year_min', 'value': year_min},
    {'check': 'year_max', 'value': year_max},
    {'check': 'duplicate_country_year_rows', 'value': dup_country_year},
])

audit_missing = df[['m2_growth','inflation','gdp_growth','trade_open','gdp_pc_growth','pop_growth','investment_share','instrument_m2_l1','instrument_m2_external_level']].isna().sum().to_frame('missing_count')
audit_missing['missing_share'] = audit_missing['missing_count'] / len(df)

forbidden_for_gdp = {'gdp_pc_growth'}
proposed_controls = {'trade_open', 'pop_growth', 'investment_share', 'gdp_pc_growth'}
leakage_flags = pd.DataFrame([
    {
        'rule': 'forbidden_controls_in_gdp_models',
        'forbidden_set': ', '.join(sorted(forbidden_for_gdp)),
        'proposed_set': ', '.join(sorted(proposed_controls)),
        'violation': bool(len(forbidden_for_gdp.intersection(proposed_controls)) > 0),
        'action': 'use_restricted_controls_excluding_gdp_pc_growth',
    }
])

restricted_controls = ['trade_open', 'pop_growth', 'investment_share']

display(audit_data)
display(audit_missing)
display(leakage_flags)

log('Data and leakage audit completed.')
audit_data.to_csv(OUT_T / 'data_audit_summary.csv', index=False)
audit_missing.reset_index().rename(columns={'index':'variable'}).to_csv(OUT_T / 'data_audit_missingness.csv', index=False)
leakage_flags.to_csv(OUT_T / 'leakage_flags.csv', index=False)


In [ ]:
def fit_fe(df_in, outcome, controls=None):
    controls = controls or []
    cols = ['Country Name', 'year', outcome, 'm2_growth'] + controls
    d = df_in[cols].dropna().copy()
    d = d.set_index(['Country Name', 'year']).sort_index()
    rhs = 'm2_growth'
    if controls:
        rhs += ' + ' + ' + '.join(controls)
    formula = f'{outcome} ~ 1 + {rhs} + EntityEffects + TimeEffects'
    res = PanelOLS.from_formula(formula, data=d).fit(cov_type='clustered', cluster_entity=True)
    return res, d


def fit_iv_twfe(df_in, outcome, instrument, controls):
    cols = ['Country Name', 'year', outcome, 'm2_growth', instrument] + controls
    d = df_in[cols].dropna().copy().rename(columns={'Country Name': 'country'})
    formula = f'{outcome} ~ 1 + ' + ' + '.join(controls) + f' + C(country) + C(year) [m2_growth ~ {instrument}]'
    res = IV2SLS.from_formula(formula, data=d).fit(cov_type='clustered', clusters=d['country'])
    return res, d


def first_stage_twfe(df_in, instrument, controls):
    cols = ['Country Name', 'year', 'm2_growth', instrument] + controls
    d = df_in[cols].dropna().copy().rename(columns={'Country Name':'country'})
    formula = 'm2_growth ~ ' + instrument + ' + ' + ' + '.join(controls) + ' + C(country) + C(year)'
    res = smf.ols(formula, data=d).fit()
    return {
        'instrument': instrument,
        'nobs': int(res.nobs),
        'coef': float(res.params[instrument]),
        't_stat': float(res.tvalues[instrument]),
        'approx_F_t2': float(res.tvalues[instrument]**2),
        'p_value': float(res.pvalues[instrument]),
    }, d


def row_model(model, outcome, spec, coef, p, nobs, extra=None):
    out = {
        'model': model,
        'outcome': outcome,
        'spec': spec,
        'coef_m2_growth': float(coef),
        'p_value_m2_growth': float(p),
        'nobs': int(nobs),
    }
    if extra:
        out.update(extra)
    return out


In [ ]:
rows = []

for outcome in ['inflation', 'gdp_growth']:
    res, d_used = fit_fe(df, outcome, controls=[])
    rows.append(row_model('fe_baseline_twfe', outcome, 'core', res.params['m2_growth'], res.pvalues['m2_growth'], res.nobs, {'r2_within': float(res.rsquared_within)}))

for outcome in ['inflation', 'gdp_growth']:
    res, d_used = fit_fe(df, outcome, controls=restricted_controls)
    rows.append(row_model('fe_controls_restricted_twfe', outcome, 'core', res.params['m2_growth'], res.pvalues['m2_growth'], res.nobs, {'r2_within': float(res.rsquared_within)}))

for outcome in ['inflation', 'gdp_growth']:
    res, d_used = fit_iv_twfe(df, outcome, instrument='instrument_m2_l1', controls=restricted_controls)
    rows.append(row_model('iv_twfe_lag', outcome, 'core', res.params['m2_growth'], res.pvalues['m2_growth'], res.nobs, {'r2': float(res.rsquared)}))

for outcome in ['inflation', 'gdp_growth']:
    res, d_used = fit_iv_twfe(df, outcome, instrument='instrument_m2_external_level', controls=restricted_controls)
    rows.append(row_model('iv_twfe_external', outcome, 'core', res.params['m2_growth'], res.pvalues['m2_growth'], res.nobs, {'r2': float(res.rsquared)}))

core_tbl = pd.DataFrame(rows)
display(core_tbl)
core_tbl.to_csv(OUT_T / 'core_model_results.csv', index=False)
log('Core model re-estimation completed.')


In [ ]:
fs_lag, fs_lag_data = first_stage_twfe(df, 'instrument_m2_l1', restricted_controls)
fs_ext, fs_ext_data = first_stage_twfe(df, 'instrument_m2_external_level', restricted_controls)
first_stage_tbl = pd.DataFrame([fs_lag, fs_ext])
display(first_stage_tbl)
first_stage_tbl.to_csv(OUT_T / 'first_stage_strength.csv', index=False)

preferred_iv = 'instrument_m2_external_level'
preferred_fs = first_stage_tbl.loc[first_stage_tbl['instrument'] == preferred_iv].iloc[0]

weak_iv_rows = []
for outcome in ['inflation', 'gdp_growth']:
    res, d_used = fit_iv_twfe(df, outcome, instrument=preferred_iv, controls=restricted_controls)
    ar_stat = None
    ar_p = None
    try:
        ar = res.anderson_rubin
        ar_stat = float(ar.stat)
        ar_p = float(ar.pval)
    except Exception:
        pass
    weak_iv_rows.append({
        'outcome': outcome,
        'preferred_iv': preferred_iv,
        'first_stage_F_proxy': float(preferred_fs['approx_F_t2']),
        'anderson_rubin_stat': ar_stat,
        'anderson_rubin_pvalue': ar_p,
        'note': 'AR may be unavailable for exactly identified specs.',
    })

weak_iv_tbl = pd.DataFrame(weak_iv_rows)
display(weak_iv_tbl)
weak_iv_tbl.to_csv(OUT_T / 'weak_iv_diagnostics.csv', index=False)
log('IV credibility diagnostics completed.')


In [ ]:
rng = np.random.default_rng(42)

lead_data = df[['Country Name', 'year', 'm2_growth', preferred_iv] + restricted_controls].dropna().copy()
lead_data = lead_data.sort_values(['Country Name', 'year']).rename(columns={'Country Name':'country'})
lead_data['instrument_lead'] = lead_data.groupby('country')[preferred_iv].shift(-1)
lead_data = lead_data.dropna(subset=['instrument_lead'])
lead_res = smf.ols('m2_growth ~ instrument_lead + ' + ' + '.join(restricted_controls) + ' + C(country) + C(year)', data=lead_data).fit()

perm_data = df[['Country Name', 'year', 'm2_growth', preferred_iv] + restricted_controls].dropna().copy().rename(columns={'Country Name':'country'})
countries = np.array(sorted(perm_data['country'].unique()))
shuffled = countries.copy()
rng.shuffle(shuffled)
map_country = dict(zip(countries, shuffled))
perm_data['country_perm'] = perm_data['country'].map(map_country)
lookup = perm_data[['country', 'year', preferred_iv]].rename(columns={'country':'country_perm', preferred_iv:'instrument_perm'})
perm_data = perm_data.merge(lookup, on=['country_perm','year'], how='left').dropna(subset=['instrument_perm'])
perm_res = smf.ols('m2_growth ~ instrument_perm + ' + ' + '.join(restricted_controls) + ' + C(country) + C(year)', data=perm_data).fit()

placebo_tbl = pd.DataFrame([
    {
        'test': 'lead_placebo',
        'coef': float(lead_res.params['instrument_lead']),
        'p_value': float(lead_res.pvalues['instrument_lead']),
        'approx_F_t2': float(lead_res.tvalues['instrument_lead']**2),
        'nobs': int(lead_res.nobs),
    },
    {
        'test': 'permutation_placebo',
        'coef': float(perm_res.params['instrument_perm']),
        'p_value': float(perm_res.pvalues['instrument_perm']),
        'approx_F_t2': float(perm_res.tvalues['instrument_perm']**2),
        'nobs': int(perm_res.nobs),
    }
])

display(placebo_tbl)
placebo_tbl.to_csv(OUT_T / 'placebo_tests.csv', index=False)
log('Placebo tests completed.')


In [ ]:
stab_rows = []
base_inf = core_tbl[(core_tbl['model']=='fe_baseline_twfe') & (core_tbl['outcome']=='inflation')].iloc[0]
baseline_coef = float(base_inf['coef_m2_growth'])

tail_cut = df['inflation'].quantile(0.99)
d_tail = df[df['inflation'] <= tail_cut].copy()
r_tail, _ = fit_fe(d_tail, 'inflation', controls=[])
stab_rows.append({'spec':'tail_exclusion_99pct','coef':float(r_tail.params['m2_growth']),'p_value':float(r_tail.pvalues['m2_growth']),'nobs':int(r_tail.nobs)})

for lo, hi, nm in [(1991,2005,'period_1991_2005'), (2006,2020,'period_2006_2020')]:
    d = df[(df['year']>=lo)&(df['year']<=hi)].copy()
    r, _ = fit_fe(d, 'inflation', controls=[])
    stab_rows.append({'spec':nm,'coef':float(r.params['m2_growth']),'p_value':float(r.pvalues['m2_growth']),'nobs':int(r.nobs)})

import requests
meta = requests.get('https://api.worldbank.org/v2/country?format=json&per_page=400', timeout=40).json()[1]
region_map = []
for m in meta:
    cc = m.get('id')
    rg = (m.get('region') or {}).get('value')
    if cc and rg:
        region_map.append((cc, rg))
region_df = pd.DataFrame(region_map, columns=['Country Code','region'])

m2raw = pd.read_csv(ROOT/'m2_raw.csv', skiprows=4)
name_code = m2raw[['Country Name','Country Code']].drop_duplicates()

d_reg = df.merge(name_code, on='Country Name', how='left').merge(region_df, on='Country Code', how='left')
region_counts = d_reg['region'].value_counts(dropna=True)
region_candidates = [x for x in region_counts.index.tolist() if x not in ['Aggregates']]
MAX_REGION_CHECKS = 2
for rg in region_candidates[:MAX_REGION_CHECKS]:
    sub = d_reg[d_reg['region'] != rg].copy()
    try:
        r, _ = fit_fe(sub, 'inflation', controls=[])
        stab_rows.append({'spec':f'leave_out_region::{rg}','coef':float(r.params['m2_growth']),'p_value':float(r.pvalues['m2_growth']),'nobs':int(r.nobs)})
    except Exception:
        continue

stab_tbl = pd.DataFrame(stab_rows)
stab_tbl['baseline_coef'] = baseline_coef
stab_tbl['abs_drift_pct'] = (stab_tbl['coef'] - baseline_coef).abs() / (abs(baseline_coef) if abs(baseline_coef) > 1e-8 else np.nan)

gate_rows = [
    {'spec':'fe_baseline_twfe','coef':float(core_tbl[(core_tbl['model']=='fe_baseline_twfe') & (core_tbl['outcome']=='inflation')]['coef_m2_growth'].iloc[0]),'p_value':float(core_tbl[(core_tbl['model']=='fe_baseline_twfe') & (core_tbl['outcome']=='inflation')]['p_value_m2_growth'].iloc[0])},
    {'spec':'fe_controls_restricted_twfe','coef':float(core_tbl[(core_tbl['model']=='fe_controls_restricted_twfe') & (core_tbl['outcome']=='inflation')]['coef_m2_growth'].iloc[0]),'p_value':float(core_tbl[(core_tbl['model']=='fe_controls_restricted_twfe') & (core_tbl['outcome']=='inflation')]['p_value_m2_growth'].iloc[0])},
    {'spec':'iv_twfe_external','coef':float(core_tbl[(core_tbl['model']=='iv_twfe_external') & (core_tbl['outcome']=='inflation')]['coef_m2_growth'].iloc[0]),'p_value':float(core_tbl[(core_tbl['model']=='iv_twfe_external') & (core_tbl['outcome']=='inflation')]['p_value_m2_growth'].iloc[0])},
    {'spec':'period_1991_2005','coef':float(stab_tbl[stab_tbl['spec']=='period_1991_2005']['coef'].iloc[0]),'p_value':float(stab_tbl[stab_tbl['spec']=='period_1991_2005']['p_value'].iloc[0])},
    {'spec':'period_2006_2020','coef':float(stab_tbl[stab_tbl['spec']=='period_2006_2020']['coef'].iloc[0]),'p_value':float(stab_tbl[stab_tbl['spec']=='period_2006_2020']['p_value'].iloc[0])},
]

gate_tbl = pd.DataFrame(gate_rows)
gate_tbl['abs_drift_pct'] = (gate_tbl['coef'] - baseline_coef).abs() / (abs(baseline_coef) if abs(baseline_coef) > 1e-8 else np.nan)

stab_tbl.to_csv(OUT_T / 'spec_stability_table.csv', index=False)
gate_tbl.to_csv(OUT_T / 'spec_gate_table.csv', index=False)

fig, ax = plt.subplots(figsize=(9,4))
plot_df = gate_tbl.copy()
sns.barplot(data=plot_df, x='spec', y='coef', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.axhline(baseline_coef, color='red', linestyle='--', label='Baseline FE coef')
ax.set_title('Inflation Coefficient Across Gate Specs')
ax.set_ylabel('Coefficient on m2_growth')
ax.tick_params(axis='x', rotation=25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_F / 'stability_coefficients_gate.png', dpi=170)

log('Stability audit completed.')
display(gate_tbl)


In [ ]:
score_rows = []

c1_val = float(preferred_fs['approx_F_t2'])
c1_pass = c1_val >= 10
score_rows.append({'criterion':'preferred_first_stage_F_ge_10','value':c1_val,'threshold':'>=10','pass':c1_pass})

positive_count = int((gate_tbl['coef'] > 0).sum())
c2_pass = positive_count >= 4
score_rows.append({'criterion':'inflation_positive_sign_at_least_4_of_5','value':positive_count,'threshold':'>=4','pass':c2_pass})

max_drift = float(gate_tbl['abs_drift_pct'].max())
c3_pass = max_drift < 0.40
score_rows.append({'criterion':'max_inflation_drift_lt_40pct','value':max_drift,'threshold':'<0.40','pass':c3_pass})

core_gdp = core_tbl[core_tbl['outcome']=='gdp_growth'].copy()
weak_count = int((core_gdp['p_value_m2_growth'] >= 0.05).sum())
all_gdp_specs = int(len(core_gdp))
c4_pass = weak_count == all_gdp_specs
score_rows.append({'criterion':'gdp_effect_weak_all_core_specs','value':f'{weak_count}/{all_gdp_specs}','threshold':'all p>=0.05','pass':c4_pass})

placebo_sig_count = int((placebo_tbl['p_value'] < 0.05).sum())
c5_pass = placebo_sig_count == 0
score_rows.append({'criterion':'placebo_tests_not_significant','value':placebo_sig_count,'threshold':'0 significant','pass':c5_pass})

scorecard = pd.DataFrame(score_rows)
all_pass = bool(scorecard['pass'].all())
recommendation = 'GO_CONTINUE' if all_pass else 'GO_PIVOT_SHORT_RUN'

scorecard['recommendation_if_fail'] = np.where(scorecard['pass'], '', 'GO_PIVOT_SHORT_RUN')
scorecard.to_csv(OUT_T / 'audit_scorecard.csv', index=False)

powerbi = core_tbl.copy()
powerbi['abs_coef'] = powerbi['coef_m2_growth'].abs()
powerbi['is_significant_5pct'] = powerbi['p_value_m2_growth'] < 0.05
powerbi['recommendation'] = recommendation
powerbi.to_csv(OUT_T / 'powerbi_model_summary.csv', index=False)

fig2, ax2 = plt.subplots(figsize=(9,4))
fs_plot = first_stage_tbl[['instrument','approx_F_t2']].copy()
fs_plot['metric'] = 'first_stage_F_t2'
pl_plot = placebo_tbl[['test','approx_F_t2']].rename(columns={'test':'instrument'})
pl_plot['metric'] = 'placebo_F_t2'
combo = pd.concat([fs_plot[['instrument','approx_F_t2','metric']], pl_plot[['instrument','approx_F_t2','metric']]], ignore_index=True)
combo = combo.rename(columns={'approx_F_t2':'value'})
sns.barplot(data=combo, x='instrument', y='value', hue='metric', ax=ax2)
ax2.axhline(10, color='red', linestyle='--', label='F=10 threshold')
ax2.set_title('First-Stage and Placebo Strength Diagnostics')
ax2.tick_params(axis='x', rotation=20)
ax2.legend()
fig2.tight_layout()
fig2.savefig(OUT_F / 'first_stage_and_placebo_strength.png', dpi=170)

log(f'Final recommendation: {recommendation}')
print('Recommendation:', recommendation)
display(scorecard)


In [ ]:
score = pd.read_csv(OUT_T / 'audit_scorecard.csv')
rec = 'GO_CONTINUE' if bool(score['pass'].all()) else 'GO_PIVOT_SHORT_RUN'

memo_path = OUT / 'phase1_audit_memo.md'
memo_lines = [
    '# Phase 1 Audit Memo',
    '',
    f'## Recommendation: `{rec}`',
    '',
    '## What was audited',
    '- Data/leakage checks (uniqueness, missingness, forbidden-variable check)',
    '- Fixed core model set (FE baseline, FE+restricted controls, IV TWFE lag, IV TWFE external)',
    '- IV credibility (first-stage strength, weak-IV notes, placebo tests)',
    '- Stability checks (period split, tail exclusion, leave-one-region-out)',
    '',
    '## Scorecard snapshot',
]
for _, r in score.iterrows():
    memo_lines.append(f"- {r['criterion']}: pass={bool(r['pass'])}, value={r['value']}, threshold={r['threshold']}")

if rec == 'GO_PIVOT_SHORT_RUN':
    memo_lines += [
        '',
        '## Why pivot now',
        '- At least one hard audit threshold failed.',
        '- Further extension on current path has lower expected value than a short-run policy-effectiveness design.',
        '',
        '## Next project definition',
        '- Objective: short-run monetary policy effectiveness (annual cross-country panel).',
        '- Method anchor: panel local projections + shock IV.',
        '- Horizons: h=0,1,2,3 for inflation and GDP growth responses.',
    ]

memo_path.write_text('\n'.join(memo_lines))

tex_path = OUT / 'phase1_audit_short_report.tex'
pdf_path = OUT / 'phase1_audit_short_report.pdf'

rows_tex = []
for _, r in score.iterrows():
    rows_tex.append(f"{r['criterion']} & {r['value']} & {r['threshold']} & {str(bool(r['pass']))} \\")

tex_lines = [
    r'\documentclass[11pt]{article}',
    r'\usepackage[margin=1in]{geometry}',
    r'\usepackage{booktabs}',
    r'\title{Phase 1 Audit Short Report}',
    r'\author{Steven Chung}',
    r'\date{\today}',
    r'\begin{document}',
    r'\maketitle',
    r'\section*{Recommendation}',
    rec,
    r'\section*{Scorecard}',
    r'\begin{tabular}{p{7.2cm}p{2.5cm}p{2.0cm}p{1.2cm}}',
    r'\toprule',
    r'Criterion & Value & Threshold & Pass \\',
    r'\midrule',
]
tex_lines.extend(rows_tex)
tex_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\section*{Interpretation}',
    r'This report is threshold-driven. If any criterion fails, the decision is GO\_PIVOT\_SHORT\_RUN.',
    r'\end{document}',
])
tex_path.write_text('\n'.join(tex_lines))

try:
    subprocess.run(['pdflatex', '-interaction=nonstopmode', '-halt-on-error', tex_path.name], cwd=str(OUT), check=True, capture_output=True, text=True)
    subprocess.run(['pdflatex', '-interaction=nonstopmode', '-halt-on-error', tex_path.name], cwd=str(OUT), check=True, capture_output=True, text=True)
except Exception as e:
    log(f'PDF build failed: {e}')

manifest = pd.DataFrame({'artifact': [
    str(OUT_T / 'audit_scorecard.csv'),
    str(OUT_T / 'spec_stability_table.csv'),
    str(OUT_T / 'powerbi_model_summary.csv'),
    str(OUT_F / 'stability_coefficients_gate.png'),
    str(OUT_F / 'first_stage_and_placebo_strength.png'),
    str(memo_path),
    str(pdf_path),
]})
manifest.to_csv(OUT / 'phase1_audit_manifest.csv', index=False)

log_md = OUT / 'phase1_audit_log.md'
log_md.write_text('\n'.join(['# Phase 1 Audit Log', ''] + [f'- {x}' for x in LOG]))

print('Memo:', memo_path)
print('PDF :', pdf_path)
print('Manifest:', OUT / 'phase1_audit_manifest.csv')


## Result Handling

- If recommendation is `GO_PIVOT_SHORT_RUN`, do not extend the current long-run branch except for archival cleanup.
- Use generated Power BI table and chart-ready outputs for presentation.
